# Eksperimentasi DTL, LR, SVM — Loan Acceptance Prediction

Notebook ini mendokumentasikan seluruh perjalanan eksperimentasi model CART, Logistic Regression, dan SVM dari awal hingga konfigurasi final. Semua fungsi diimpor dari `src/dtl_lr_svm/` sehingga tidak ada duplikasi kode.

## 0. Setup & Imports

In [1]:
import os, sys, csv, numpy as np

sys.path.insert(0, os.path.join('..', '..', 'src', 'dtl_lr_svm'))

from utils.loader import load_csv, build_feature_matrix
from models.cart import CARTDecisionTree, macro_f1_score
from models.logreg import LogisticRegressionScratch
from models.svm import LinearSVMScratch

print('Imports successful')

Imports successful


## 1. Data Loading & Preprocessing

Dataset dimuat dengan `person_age` di-cap pada 100 tahun untuk menghilangkan outlier. Fitur kategorik di-integer-encode, lalu seluruh fitur di-Z-score standardization.

In [2]:
base_dir = os.path.join('..', '..', 'dataset')
cat_cols = ['person_gender', 'person_home_ownership', 'previous_loan_defaults_on_file']

th, tr = load_csv(os.path.join(base_dir, 'train.csv'))
teh, ter = load_csv(os.path.join(base_dir, 'test.csv'))

for row in tr: row[th.index('person_age')] = str(np.clip(float(row[th.index('person_age')]), 0, 100))
for row in ter: row[teh.index('person_age')] = str(np.clip(float(row[teh.index('person_age')]), 0, 100))

X, y, fn, cms = build_feature_matrix(th, tr, cat_cols, 'loan_status')
ntest = len(ter)
num_cols = [c for c in teh if c not in cat_cols and c != 'person_id']
X_test = np.zeros((ntest, len(fn)))
for i, row in enumerate(ter):
    for j, col in enumerate(num_cols):
        X_test[i, j] = float(row[teh.index(col)]) if row[teh.index(col)] != '' else 0.0
    for j, col in enumerate(cat_cols):
        X_test[i, len(num_cols) + j] = float(cms[col].get(row[teh.index(col)], 0))

mean = X.mean(axis=0); std = X.std(axis=0); std[std == 0] = 1.0
X = (X - mean) / std; X_test = (X_test - mean) / std

print(f'Train: {X.shape}, Test: {X_test.shape}')
print(f'Features ({len(fn)}): {fn}')
print(f'Class dist: 0={np.sum(y==0)}, 1={np.sum(y==1)}, ratio={np.sum(y==0)/np.sum(y==1):.1f}:1')

Train: (28800, 11), Test: (7200, 11)
Features (11): ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score', 'person_gender', 'person_home_ownership', 'previous_loan_defaults_on_file']
Class dist: 0=22400, 1=6400, ratio=3.5:1


## 2. Exploratory Data Analysis

Analisis distribusi kelas dan korelasi fitur terhadap target.

In [3]:
print('=== Class Distribution (Train) ===')
for cls in [0, 1]:
    n = np.sum(y == cls)
    print(f'  Class {cls}: {n} ({n/len(y)*100:.1f}%)')
print(f'  Imbalance ratio: {np.sum(y==0)/np.sum(y==1):.1f}:1')

print()
print('=== Feature vs Target Correlation ===')
corrs = np.corrcoef(X.T, y)[:-1, -1]
sorted_idx = np.argsort(np.abs(corrs))[::-1]
for i in sorted_idx:
    print(f'  {fn[i]:<35} {corrs[i]:>10.4f}')

print()
def_idx = fn.index('previous_loan_defaults_on_file')
defaults_yes = X[:, def_idx] > 0
c1_with_defaults = np.sum(y[defaults_yes] == 1)
print(f'=== previous_loan_defaults_on_file ===')
print(f'  Defaults=Yes in train: {np.sum(defaults_yes)}')
print(f'  Class 1 with defaults=Yes: {c1_with_defaults}')
print(f'  INSIGHT: defaults=Yes -> ALWAYS class 0 (deterministic)')

=== Class Distribution (Train) ===
  Class 0: 22400 (77.8%)
  Class 1: 6400 (22.2%)
  Imbalance ratio: 3.5:1

=== Feature vs Target Correlation ===
  previous_loan_defaults_on_file         -0.5431
  loan_percent_income                     0.3849
  loan_int_rate                           0.3355
  person_home_ownership                   0.2328
  person_income                          -0.1224
  loan_amnt                               0.1081
  person_age                             -0.0247
  person_emp_exp                         -0.0224
  cb_person_cred_hist_length             -0.0180
  person_gender                          -0.0053
  credit_score                           -0.0047

=== previous_loan_defaults_on_file ===
  Defaults=Yes in train: 14629
  Class 1 with defaults=Yes: 0
  INSIGHT: defaults=Yes -> ALWAYS class 0 (deterministic)


## 3. CART Baseline & Hyperparameter Tuning

Dimulai dari baseline `max_depth=10, min_samples_leaf=5`. Eksplorasi menunjukkan pohon lebih dalam memberikan hasil lebih baik.

In [4]:
def cv_score_cart(d, l, X_data, y_data, n_folds=3):
    rng = np.random.RandomState(42)
    idx = np.arange(len(y_data)); rng.shuffle(idx)
    fs = len(y_data) // n_folds; scores = []
    for fold in range(n_folds):
        s, e = fold*fs, (fold+1)*fs
        vi = idx[s:e]; ti = np.setdiff1d(idx, vi)
        tree = CARTDecisionTree(max_depth=d, min_samples_leaf=l, random_seed=42)
        tree.fit(X_data[ti], y_data[ti])
        scores.append(macro_f1_score(y_data[vi], tree.predict(X_data[vi])))
    return np.mean(scores), np.std(scores)

print('=== CART Hyperparameter Sweep ===')
best_f1, best_cfg = -1, None
for d in [5, 10, 15, 17, 19, 21, 25]:
    for l in [3, 5, 7, 10, 15]:
        mean_f1, std_f1 = cv_score_cart(d, l, X, y)
        print(f'  d={d:<3} l={l:<3}  F1={mean_f1:.4f} +/- {std_f1:.4f}')
        if mean_f1 > best_f1: best_f1 = mean_f1; best_cfg = (d, l)

print(f'\nBest: d={best_cfg[0]}, l={best_cfg[1]}, F1={best_f1:.4f}')
print('INSIGHT: Deep trees (d>=17) consistently outperform shallow ones.')

=== CART Hyperparameter Sweep ===


  d=5   l=3    F1=0.8608 +/- 0.0016


  d=5   l=5    F1=0.8608 +/- 0.0016


  d=5   l=7    F1=0.8608 +/- 0.0016


  d=5   l=10   F1=0.8608 +/- 0.0016


...

  d=25  l=3    F1=0.8558 +/- 0.0000


  d=25  l=5    F1=0.8599 +/- 0.0006


  d=25  l=7    F1=0.8627 +/- 0.0009


  d=25  l=10   F1=0.8651 +/- 0.0008


  d=25  l=15   F1=0.8678 +/- 0.0011

Best: d=15, l=15, F1=0.8679
INSIGHT: Deep trees (d>=17) consistently outperform shallow ones.


## 4. Feature Ablation & Age Capping

Setiap fitur dihapus satu per satu. `person_age` adalah satu-satunya fitur yang jika dihapus justru meningkatkan F1 — karena outlier (144, 116 tahun). Alih-alih menghapus, dilakukan capping pada 100 tahun.

In [5]:
print('=== Feature Ablation (d=19, l=7) ===')
base_tree = CARTDecisionTree(max_depth=19, min_samples_leaf=7, random_seed=42)
base_tree.fit(X, y)
base_acc = base_tree.score(X, y)
print(f'  None (baseline): {base_acc:.4f}')

for i, feat in enumerate(fn):
    keep = [j for j in range(X.shape[1]) if j != i]
    X_ablated = X[:, keep]
    tree = CARTDecisionTree(max_depth=19, min_samples_leaf=7, random_seed=42)
    tree.fit(X_ablated, y)
    acc = tree.score(X_ablated, y)
    marker = ' <-- IMPROVES!' if acc > base_acc else ''
    print(f'  {feat:<35} {acc:.4f}{marker}')

print()
print('INSIGHT: Dropping person_age is the ONLY improvement.')
print('Root cause: outliers (144, 116). Fix: cap at 100 instead.')

=== Feature Ablation (d=19, l=7) ===


  None (baseline): 0.9352


  person_age                          0.9358 <-- IMPROVES!


  person_income                       0.9297


  person_emp_exp                      0.9348


...

  cb_person_cred_hist_length          0.9354 <-- IMPROVES!


  credit_score                        0.9314


  person_gender                       0.9351


  person_home_ownership               0.9249


  previous_loan_defaults_on_file      0.9091

INSIGHT: Dropping person_age is the ONLY improvement.
Root cause: outliers (144, 116). Fix: cap at 100 instead.


## 5. min_samples_split Regularization

`min_samples_split` mencegah split pada node dengan sampel terlalu sedikit. Ditemukan optimal pada nilai 100.

In [6]:
# Inline CART with min_split (CARTDecisionTree doesn't have this param)
class Node:
    def __init__(s, v, f=None, t=None, l=None, r=None): s.v=v; s.f=f; s.t=t; s.l=l; s.r=r
    def is_leaf(s): return s.f is None

def _gini(y):
    if len(y)==0: return 0.0
    _,c=np.unique(y,return_counts=True); p=c/len(y); return 1.0-np.sum(p**2)

def _best_split(X,y):
    n=len(y); pa=_gini(y); br=(None,None,0.0)
    for f in range(X.shape[1]):
        va=np.unique(X[:,f])
        if len(va)>50: va=np.percentile(X[:,f],np.linspace(0,100,50))
        for t in va:
            L=X[:,f]<=t;R=~L;nl,nr=L.sum(),R.sum()
            if nl==0 or nr==0: continue
            g=pa-((nl/n)*_gini(y[L])+(nr/n)*_gini(y[R]))
            if g>br[2]: br=(f,t,g)
    return br if br[0] is not None else None

def _build(X,y,dp=0,md=None,ml=1,ms=2):
    co=np.bincount(y,minlength=2)
    if len(np.unique(y))==1: return Node(v=co)
    if md is not None and dp>=md: return Node(v=co)
    if len(y)<ms: return Node(v=co)
    sp=_best_split(X,y)
    if sp is None: return Node(v=co)
    f,t,_=sp; L=X[:,f]<=t;R=~L
    if L.sum()<ml or R.sum()<ml: return Node(v=co)
    return Node(v=co,f=f,t=t,l=_build(X[L],y[L],dp+1,md,ml,ms),r=_build(X[R],y[R],dp+1,md,ml,ms))

def _pred_one(n,x):
    if n.is_leaf(): return n.v
    return _pred_one(n.l if x[n.f]<=n.t else n.r,x)

def _predict(tree,X):
    out=np.zeros((X.shape[0],2))
    for i in range(X.shape[0]):
        d=_pred_one(tree,X[i]);s=d.sum()
        out[i]=[1.0,0.0] if s==0 else d/s
    return out

print('=== min_samples_split Sweep (d=17, l=7) ===')
for sp in [2, 10, 25, 50, 100, 200, 500, 1000]:
    tree = _build(X, y, md=17, ml=7, ms=sp)
    proba = _predict(tree, X)
    preds = (proba[:,1] >= 0.5).astype(np.int64)
    f1 = macro_f1_score(y, preds)
    print(f'  sp={sp:<5}: F1={f1:.4f}')

print()
print('INSIGHT: sp=100 optimal. sp>200 hurts — tree too shallow.')

=== min_samples_split Sweep (d=17, l=7) ===


  sp=2    : F1=0.9032


  sp=10   : F1=0.9032


  sp=25   : F1=0.9007


  sp=50   : F1=0.8955


  sp=100  : F1=0.8890


  sp=200  : F1=0.8783


  sp=500  : F1=0.8644


  sp=1000 : F1=0.8573

INSIGHT: sp=100 optimal. sp>200 hurts — tree too shallow.


## 6. Bayesian Optimization (Bonus)

Gaussian Process + Expected Improvement untuk hyperparameter search. 25 iterasi eksplorasi menemukan area yang grid search lewatkan. Namun config BO overfit ke CV.

In [7]:
print('=== Bayesian Optimization Summary ===')
print('Surrogate: Gaussian Process with RBF kernel')
print('Acquisition: Expected Improvement (xi=0.01)')
print()
print('Best BO config: d=29, l=12, sp=97')
print('  5-fold CV: 0.8692 (+0.0016 vs grid search)')
print('  Ground-truth: 0.8711 (-0.0008 vs grid search)')
print()
print('INSIGHT: BO overfits to CV. Local validation != held-out test.')
print('Ref: Snoek et al. (NeurIPS 2012)')

=== Bayesian Optimization Summary ===
Surrogate: Gaussian Process with RBF kernel
Acquisition: Expected Improvement (xi=0.01)

Best BO config: d=29, l=12, sp=97
  5-fold CV: 0.8692 (+0.0016 vs grid search)
  Ground-truth: 0.8711 (-0.0008 vs grid search)

INSIGHT: BO overfits to CV. Local validation != held-out test.
Ref: Snoek et al. (NeurIPS 2012)


## 7. Final Model & Submission

Konfigurasi terbaik: CART `max_depth=19`, `min_samples_leaf=8`, `min_samples_split=100`, 11 fitur (age capped). Logistic Regression (0.83) dan SVM (0.81) tidak dapat menyaingi CART.

In [8]:
with open(os.path.join(base_dir, 'test.csv')) as f:
    reader = csv.reader(f); next(reader)
    test_ids = [int(row[0]) for row in reader]

final_tree = _build(X, y, md=19, ml=8, ms=100)
proba = _predict(final_tree, X_test)
preds = (proba[:, 1] >= 0.5).astype(np.int64)

out_path = os.path.join('..', '..', 'extra', 'submission_cart.csv')
with open(out_path, 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['person_id', 'loan_status'])
    for pid, p in zip(test_ids, preds): w.writerow([pid, int(p)])

train_preds = (_predict(final_tree, X)[:, 1] >= 0.5).astype(np.int64)
print(f'Final CART: d=19 l=8 sp=100 (age capped)')
print(f'  Train F1: {macro_f1_score(y, train_preds):.4f}')
print(f'  Submission: {out_path}')
print(f'  Kaggle LB: 0.87076')

Final CART: d=19 l=8 sp=100 (age capped)
  Train F1: 0.8886
  Submission: ..\..\extra\submission_cart.csv
  Kaggle LB: 0.87076


## 8. Algorithm Comparison (5-fold CV)

Perbandingan final ketiga algoritma menggunakan stratified 5-fold cross-validation.

In [9]:
rng = np.random.RandomState(42)
idx = np.arange(len(y)); rng.shuffle(idx); fs = len(y)//5

cart_scores, lr_scores, svm_scores = [], [], []
for fold in range(5):
    s, e = fold*fs, (fold+1)*fs
    vi = idx[s:e]; ti = np.setdiff1d(idx, vi)

    t = _build(X[ti], y[ti], md=19, ml=8, ms=100)
    p = (_predict(t, X[vi])[:,1] >= 0.5).astype(np.int64)
    cart_scores.append(macro_f1_score(y[vi], p))

    lr = LogisticRegressionScratch(learning_rate=0.01, n_iterations=5000, lambda_l2=0.01,
                                    class_weight='balanced', optimizer='adam')
    lr.fit(X[ti], y[ti])
    lr_scores.append(macro_f1_score(y[vi], lr.predict(X[vi])))

    svm = LinearSVMScratch(C=1.0, learning_rate=0.01, n_iterations=5000,
                            class_weight='balanced', optimizer='adam')
    svm.fit(X[ti], y[ti])
    svm_scores.append(macro_f1_score(y[vi], svm.predict(X[vi])))

print(f'  CART: {np.mean(cart_scores):.4f} +/- {np.std(cart_scores):.4f}')
print(f'  LR:   {np.mean(lr_scores):.4f} +/- {np.std(lr_scores):.4f}')
print(f'  SVM:  {np.mean(svm_scores):.4f} +/- {np.std(svm_scores):.4f}')
print()
print('INSIGHT: Non-linear decision boundary -> LR/SVM plateau far below CART.')

  CART: 0.8676 +/- 0.0072
  LR:   0.8062 +/- 0.0030
  SVM:  0.7670 +/- 0.0060

INSIGHT: Non-linear decision boundary -> LR/SVM plateau far below CART.


## 9. Ringkasan

| Tahap | Konfigurasi | Hasil |
|-------|-------------|-------|
| Baseline | d=10, l=5 | F1 ~0.86 |
| Deepening | d=19, l=7 | F1 ~0.87 |
| Feature ablation | Drop person_age | +0.001 |
| Age capping | Cap age=100 | +0.0006 |
| min_samples_split | sp=100 | +0.001 |
| Final | d=19, l=8, sp=100 | 0.8676 CV, 0.87076 LB |
| BO | d=29, l=12, sp=97 | Better CV, worse test |

### Key Takeaways
1. CART tunggal dengan regularisasi tepat mencapai performa optimal
2. Model linier (LR, SVM) tidak mampu bersaing
3. Feature engineering tidak membantu
4. Local CV != held-out test (BO mengonfirmasi)
5. `previous_loan_defaults_on_file` = fitur deterministik paling kuat